# 2025-26 Player Coverage — Building the Season Config

**Ticket:** #30 — Coverage workflow for building the 2025-26 config  
**Deliverable:** `config/2025-26/players.yaml`

## Method — discourse-driven, not a per-team quota

Track the *individually-discussed* players, not every roster slot. A discourse-thin
market with one nationally-discussed star legitimately has one row — that's accurate,
not a gap. The question is **"are we missing players who actually generate r/NBA
discourse?"**, not "does every team have ≥2?"

This notebook answers **two distinct questions** (see ticket); only the first is
local-data work:

1. **What did the v1 method *miss* in 2024-25?** (this notebook, §1) — mine 2024-25
   `sentiment.parquet` for `sentiment_player` values the v1 alias map never resolved:
   the discussed-but-untracked candidates.
2. **What's *new* for 2025-26?** (rookies, breakouts, trade movers) — sourced from an
   authoritative roster source, handled separately (a wrong team mis-attributes a whole
   fanbase).

## Guardrails

- **`sentiment.parquet` access is aggregate-only — the `body` column is never read.**
  The DuckDB scan below names only `sentiment_player`, so `body` never leaves disk
  (projection pushdown).
- The resolution test **reuses the pipeline's own `resolve_sentiment_player()`**
  (`utils.player_config`) — the same resolver `resolve_player` uses for attribution —
  so coverage and production share one implementation, with no alias logic re-written here.

In [1]:
import os
import sys
from pathlib import Path

# Bootstrap: run from anywhere. Jupyter sets the kernel cwd to this notebook's
# folder (notebooks/2025-26/), so walk up to the project root (the dir with
# pyproject.toml), chdir there (utils.paths resolves ./data relative to cwd),
# and put it on sys.path (so `import utils` works without an editable install).
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import duckdb
import polars as pl

from utils.paths import get_processed_dir, get_dashboard_dir
from utils.player_config import build_alias_to_player_map, resolve_sentiment_player
from utils.season_config import get_active_season

SEASON = get_active_season()
SENTIMENT_PARQUET = get_processed_dir() / "sentiment.parquet"
PLAYER_METADATA = get_dashboard_dir() / "player_metadata.parquet"

# The v1 alias map — the config that WAS used to classify 2024-25.
# A sentiment_player "miss" is a value resolve_sentiment_player can't resolve.
alias_map = build_alias_to_player_map()
canonical_names = set(alias_map.values())

print(f"Project root:      {_root}")
print(f"Active season:     {SEASON}")
print(f"Sentiment parquet: {SENTIMENT_PARQUET}")
print(f"Alias map:         {len(alias_map):,} aliases -> {len(canonical_names)} tracked players")

Project root:      /Users/oluobiri/Documents/nba-hate-tracker
Active season:     2024-25
Sentiment parquet: data/2024-25/processed/sentiment.parquet
Alias map:         455 aliases -> 111 tracked players


## 1. Aggregate scan — `sentiment_player` volume (no bodies)

`sentiment_player` is the classifier's free-text pick of who each comment is about,
*regardless* of whether the v1 config could resolve that name. Its volume distribution
is our signal for discussed-but-untracked players.

DuckDB reads the parquet directly and the query names **only** `sentiment_player` — so
`body` is never scanned. `.pl()` hands the small aggregate result to polars/Python for
the resolution filter below.

In [2]:
# DuckDB reads only the sentiment_player column from the parquet (projection
# pushdown). The `body` column is never touched.
QUERY = f"""
    SELECT sentiment_player, count(*) AS mentions
    FROM read_parquet('{SENTIMENT_PARQUET}')
    WHERE sentiment_player IS NOT NULL
    GROUP BY sentiment_player
    ORDER BY mentions DESC
"""
sp_counts = duckdb.sql(QUERY).pl()

print(f"Distinct non-null sentiment_player values: {sp_counts.height:,}")
print(f"Total attributed mentions:                 {sp_counts['mentions'].sum():,}")

Distinct non-null sentiment_player values: 9,792
Total attributed mentions:                 1,224,457


## 1.2 Precise miss filter — the production resolver

The raw "`sentiment_player` not in canonical names" check **over-reports**: the model
writes "Shai", "SGA", or `Michael Porter Jr.` (trailing period), which all map to tracked
players. We resolve each distinct value through the pipeline's own
`resolve_sentiment_player()` — which normalizes punctuation/case before the alias
lookup — and keep only the names that return `None` (genuinely untracked).

Because coverage and production call the **same** resolver, a punctuation variant that
resolves here also attributes correctly in the v2 run. (We resolve over the ~10k-row
aggregate, not the 1.93M-row fact, so a plain Python loop here is trivial — this is *not*
the row-wise-on-the-full-frame anti-pattern.)

In [3]:
# Resolve each distinct sentiment_player via the SAME function the pipeline uses
# for attribution. A miss = resolves to None (no tracked player).
resolved = [resolve_sentiment_player(n, alias_map) for n in sp_counts["sentiment_player"].to_list()]
misses = sp_counts.filter(pl.Series("_resolved", resolved).is_null())

print(f"Unresolved (discussed-but-untracked) values: {misses.height:,}")
print(f"  total mentions in misses: {misses['mentions'].sum():,}")
print()
with pl.Config(tbl_rows=40):
    print(misses.head(40))

Unresolved (discussed-but-untracked) values: 9,403
  total mentions in misses: 117,929

shape: (40, 2)
┌───────────────────────┬──────────┐
│ sentiment_player      ┆ mentions │
│ ---                   ┆ ---      │
│ str                   ┆ i64      │
╞═══════════════════════╪══════════╡
│ Murray                ┆ 5165     │
│ Dwight Howard         ┆ 2651     │
│ JB                    ┆ 2330     │
│ Carmelo Anthony       ┆ 2248     │
│ Michael Jordan        ┆ 2178     │
│ Nico Harrison         ┆ 1958     │
│ Derrick Rose          ┆ 1710     │
│ Green                 ┆ 1688     │
│ Kobe Bryant           ┆ 1635     │
│ Nico                  ┆ 1556     │
│ Shaq                  ┆ 1057     │
│ Thompson              ┆ 931      │
│ JJ Redick             ┆ 925      │
│ Melo                  ┆ 845      │
│ Stephen A. Smith      ┆ 822      │
│ KPJ                   ┆ 819      │
│ Paul Pierce           ┆ 799      │
│ John Wall             ┆ 642      │
│ Bill Russell          ┆ 640      │
│ Kevin H

## 1.3 Triage (next) — threshold & 2025-26 relevance

The ranked misses above are 2024-25-discussed names. Before any becomes a 2025-26
config row, each must clear two gates:

- **Volume threshold** — enough mentions for a stable rate (notebook 06 used ≈200–250
  as a reference point; discourse-driven, not an absolute cutoff).
- **Still relevant in 2025-26?** — retired / out-of-league / non-players (GMs, coaches,
  media, legends) drop out here. This is where Question 1 (v1 misses) hands off to
  Question 2 (the 2025-26 roster build).

Note: punctuation-variant misses (e.g. `Michael Porter Jr.`) are no longer in this
list — `resolve_sentiment_player()` now normalizes them upstream (which also fixed a
latent attribution drop in `resolve_player`). What remains is genuine coverage gaps
(e.g. **KPJ** = Kevin Porter Jr., **Braun**, **Flagg**, **Podz**) mixed with non-players.

**Known over-report trap** (from notebook 06 §1.2): alias *contamination* in the body
match — e.g. "Aaron Wiggins" vs "Andrew Wiggins", "Harrison Barnes" vs "Scottie Barnes".
Alias verification (the `nba-superfan` agent, ticket step 3) is the chosen mechanism for
these — not body sampling, which the aggregate-only guardrail rules out this session.

In [4]:
# Focus the miss list to a discourse-relevant volume floor. The threshold is a
# *focusing* device (notebook 06 used ~200), not a quota: below it, sentiment
# rates are too thin to read. We deliberately do NOT auto-classify
# legend / non-player / ambiguous-surname — there is no robust textual tell that
# separates them from real gaps, so that call is left to human judgment over
# this short list (the discourse-driven decision).
THRESHOLD = 200
pool = misses.filter(pl.col("mentions") >= THRESHOLD)
print(f"Misses with >= {THRESHOLD} mentions: {pool.height} (of {misses.height})")
with pl.Config(tbl_rows=-1):
    print(pool)

Misses with >= 200 mentions: 104 (of 9403)
shape: (104, 2)
┌───────────────────────┬──────────┐
│ sentiment_player      ┆ mentions │
│ ---                   ┆ ---      │
│ str                   ┆ i64      │
╞═══════════════════════╪══════════╡
│ Murray                ┆ 5165     │
│ Dwight Howard         ┆ 2651     │
│ JB                    ┆ 2330     │
│ Carmelo Anthony       ┆ 2248     │
│ Michael Jordan        ┆ 2178     │
│ Nico Harrison         ┆ 1958     │
│ Derrick Rose          ┆ 1710     │
│ Green                 ┆ 1688     │
│ Kobe Bryant           ┆ 1635     │
│ Nico                  ┆ 1556     │
│ Shaq                  ┆ 1057     │
│ Thompson              ┆ 931      │
│ JJ Redick             ┆ 925      │
│ Melo                  ┆ 845      │
│ Stephen A. Smith      ┆ 822      │
│ KPJ                   ┆ 819      │
│ Paul Pierce           ┆ 799      │
│ John Wall             ┆ 642      │
│ Bill Russell          ┆ 640      │
│ Kevin Hart            ┆ 622      │
│ Pat Beverley  

## 1.4 Step-1 handoff — the candidate set

Reading the ≥200 pool by judgment (legends / non-players / ambiguous surnames filtered
by eye, not code) yields the actionable lists below, recorded as the structured handoff
to **Step 2** (authoritative team assignment — *never* model memory) and **Step 3**
(`nba-superfan` alias verification + collision checks).

Teams are intentionally omitted here. Co-mention clustering recovered three
single-referent surname/initial aliases and surfaced one new player (Jaxson Hayes);
`jb` was **dropped** as ambiguous (Jaylen Brown vs Jalen Brunson — both stars, so the
2-letter alias is unsafe regardless of which way v1 leans). Miles Bridges sits below the
discourse floor (~70 mentions) — a watch item for Step-2 judgment, not a v1-driven add.

The handoff also carries a **`short_aliases` sync** sub-task for Step 3 (the list is
hand-synced with per-player aliases — state-doc §3.7): **remove 9 orphans** that are in
`short_aliases` but bound to no player (`james` is a leftover from the v1.2 Harden-
contamination fix), and **add** the new short tokens (`ag`, `ar`, `brooks`) so they get
word-boundary matching rather than substring-flooding.

In [5]:
# Step-1 output. Teams are omitted — assigned in Step 2 from an authoritative roster
# source. Notes carry identification hints only, not the roster_team field.

# 1) NEW untracked players to add. v1 mentions = the discourse signal.
#    tier "clear" = strong standalone discourse; "verify" = role-player / spike-driven,
#    confirm 2025-26 relevance in Step 2.
NEW_PLAYERS = [
    # (canonical_name, v1_mentions, tier, note)
    ("Stephon Castle", 543, "clear", "Spurs ROY; 'Castle' + full name"),
    ("Kevin Porter Jr", 819, "clear", "KPJ; distinct from Michael Porter Jr"),
    ("Andrew Nembhard", 527, "verify", "spike-driven via Pacers playoff run"),
    ("Christian Braun", 442, "clear", "contract-extension discourse"),
    ("Cooper Flagg", 433, "verify", "2025 #1 pick; v1 vol is pre-draft hype"),
    ("Aaron Nesmith", 420, "clear", ""),
    ("Brandin Podziemski", 412, "clear", "polarized GSW discourse"),
    ("Kyle Kuzma", 410, "clear", ""),
    ("Davion Mitchell", 381, "clear", ""),
    ("Kentavious Caldwell-Pope", 349, "clear", "KCP"),
    ("Dwight Powell", 346, "verify", ""),
    ("Isaiah Hartenstein", 332, "clear", ""),
    ("Steven Adams", 331, "clear", ""),
    ("Bennedict Mathurin", 331, "clear", ""),
    ("Jaxson Hayes", 303, "clear", "surfaced via 'Hayes' co-mention (Lakers cluster)"),
    ("Jaylen Wells", 299, "verify", "Grizzlies rookie"),
    ("Zach Collins", 277, "verify", ""),
    ("TJ McConnell", 276, "clear", ""),
    ("AJ Green", 270, "verify", "Bucks"),
    ("Luka Garza", 256, "verify", ""),
    ("Dereck Lively", 253, "clear", ""),
    ("Jaden McDaniels", 238, "clear", ""),
    ("Obi Toppin", 217, "verify", ""),
    ("RJ Barrett", 215, "clear", ""),
    ("Ajay Mitchell", 213, "verify", "OKC rookie"),
    ("Max Christie", 207, "verify", ""),
]

# 2) Alias/spelling gaps on ALREADY-TRACKED players (player is in the config; the
#    model's spelling/format dodged the alias map). Add aliases in Step 3.
ALIAS_GAPS = {
    # tracked_player: (alias_to_add, v1_missed, reason)
    "Jalen Williams": ("jdub", 594, "config has 'j-dub'; model writes 'JDub'/'Jdub'"),
    "Tyrese Haliburton": ("halliburton", 475, "misspelling, double-L"),
    "Jayson Tatum": ("jason tatum", 361, "misspelling 'Jason'"),
    "Marvin Bagley III": ("marvin bagley", 219, "suffix-less form"),
    "Luka Doncic": ("dončić", 225, "diacritic form 'Dončić'"),
}

# 3) Surname/initial recoveries — single referent by co-mention cluster. Add as an
#    alias for the EXISTING tracked player, pending Step-3 collision check (short /
#    common tokens are collision-prone; longest-first + word-boundary discipline).
SURNAME_RECOVERIES = {
    # tracked_player: (alias, v1_vol, evidence, risk)
    "Dillon Brooks": ("brooks", 397, "GSW-beef cluster (Curry/Draymond); only tracked Brooks", "low"),
    "Aaron Gordon": ("ag", 305, "Nuggets cluster (Jokic/Westbrook/MPJ)", "2-letter"),
    "Austin Reaves": ("ar", 259, "Lakers cluster (LeBron/Luka/AD)", "2-letter"),
}
# Dropped: 'jb' (Jaylen Brown vs Jalen Brunson — ambiguous between two stars).
# Dropped surnames (genuinely split by co-mention): Murray, Green, Thompson, James,
# Paul, Russell, Jalen.

# 4) short_aliases sync (Step 3) — short_aliases flags which aliases get word-boundary
#    matching; it is hand-synced with per-player aliases (a known seam, state-doc §3.7).
#    a) REMOVE orphans: in short_aliases but bound to NO player (inert leftovers — note
#       several are exactly the ambiguous surnames the config intentionally never bound).
SHORT_ALIAS_ORPHANS = ["ai", "brown", "green", "james", "jordan", "murray", "paul", "smart", "tt"]
#    b) ADD for the new short recoveries above — 2-letter/common tokens MUST be flagged
#       for word-boundary matching or they substring-match ('ag' in "again", 'ar' in "are").
SHORT_ALIAS_ADDITIONS = ["ag", "ar", "brooks"]

# Below the discourse floor — NOT added from v1; Step-2 may override on expectation.
WATCH = {"Miles Bridges": 70}  # Miles+Myles forms; Hornets, controversy peaked 2022

_clear = sum(1 for p in NEW_PLAYERS if p[2] == "clear")
print(f"New players:          {len(NEW_PLAYERS)}  ({_clear} clear, {len(NEW_PLAYERS) - _clear} verify)")
print(f"Alias gaps (tracked): {len(ALIAS_GAPS)}  (~{sum(v[1] for v in ALIAS_GAPS.values()):,} missed comments)")
print(f"Surname recoveries:   {len(SURNAME_RECOVERIES)}  (jb dropped as ambiguous)")
print(f"short_aliases sync:   -{len(SHORT_ALIAS_ORPHANS)} orphans, +{len(SHORT_ALIAS_ADDITIONS)} new")
print(f"Watch (sub-floor):    {WATCH}")

New players:          26  (16 clear, 10 verify)
Alias gaps (tracked): 5  (~1,874 missed comments)
Surname recoveries:   3  (jb dropped as ambiguous)
short_aliases sync:   -9 orphans, +3 new
Watch (sub-floor):    {'Miles Bridges': 70}


## 2. Building the 2025-26 config

Step 1 found who to track. Step 2 gives each player a team, sourced from `nba_api`
rather than memory — a wrong team mis-attributes a whole fanbase.

The split: `nba_api` owns the factual fields (team, `player_id`, experience); the
config owns the curated layer (tracked set + aliases). They join on `player_id`,
stable across the diacritic/"Jr." drift that broke name matching in §1.2.

We bake the factual fields into `players.yaml` (what the pipeline reads today).
Thinning the config to curation-only and joining facts at build time is the open
PR #44 question.

Inclusion is discourse-driven, not roster-gated: nothing is dropped here. The
"worth classifying" prune happens later, after filtering, on real comment volume.

- 2.1 — pull and cache the roster snapshot
- 2.2 — reconcile carry-overs (same / moved / off-roster)
- 2.3 — additions (mined + rookies/young + veterans)
- 2.4 — assemble and write the config

> **Superseded (2026-07-29):** the §2.1 acquisition cell below is kept as historical record; the roster snapshot is now fetched by `uv run python -m scripts.fetch_rosters` (`pipeline/nba_stats.py`, #39), which also captures `height`/`weight`.

In [6]:
# §2.1 — Authoritative 2025-26 roster snapshot (nba_api commonteamroster).
# Cached as a reference asset; every column is from nba_api, normalized to
# lowercase snake_case so it joins the Player dimension USING (player_id).
# Re-running is cheap: if the snapshot exists we load it instead of re-hitting
# stats.nba.com. (Graduates to a pipeline/ client once it feeds a committed
# artifact — PR #44, or v3 game data via the scoreboard endpoint.)
import time

from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.static import teams as static_teams

from utils.paths import get_data_dir

REF_DIR = get_data_dir(season="2025-26") / "reference"
REF_DIR.mkdir(parents=True, exist_ok=True)
ROSTERS_PARQUET = REF_DIR / "rosters.parquet"

# raw nba_api column -> normalized name (team_name / team_abbr added per team)
_RENAME = {
    "PLAYER_ID": "player_id",
    "PLAYER": "player_name",
    "NUM": "jersey_number",
    "POSITION": "position",
    "AGE": "age",
    "EXP": "experience",
    "BIRTH_DATE": "birth_date",
    "SCHOOL": "school",
}
_COLS = [
    "player_id", "player_name", "team_name", "team_abbr", "jersey_number",
    "position", "age", "experience", "birth_date", "school",
]


def _pull_rosters() -> pl.DataFrame:
    """Pull all 30 rosters, normalize types, cache to ROSTERS_PARQUET."""
    frames, failures = [], []
    for team in static_teams.get_teams():
        abbr, name = team["abbreviation"], team["full_name"]
        for attempt in (1, 2):
            try:
                df = commonteamroster.CommonTeamRoster(
                    team_id=team["id"], season="2025-26", timeout=30
                ).get_data_frames()[0]
                cols = [c for c in _RENAME if c in df.columns]
                frames.append(
                    pl.from_pandas(df[cols])
                    .rename({c: _RENAME[c] for c in cols})
                    .with_columns(
                        pl.lit(name).alias("team_name"),
                        pl.lit(abbr).alias("team_abbr"),
                    )
                )
                break
            except Exception as e:  # report-and-continue
                if attempt == 2:
                    failures.append((abbr, type(e).__name__))
                else:
                    time.sleep(2)
        time.sleep(0.6)  # be polite to stats.nba.com
    if failures:
        print(f"WARNING failures: {failures}")
    out = (
        pl.concat(frames, how="diagonal")
        .with_columns(
            pl.col("age").cast(pl.Int64),
            # nba_api gives "MAR 03, 1998"; titlecase so %b parses.
            pl.col("birth_date").str.to_titlecase().str.to_date("%b %d, %Y", strict=False),
        )
        .select(_COLS)
    )
    out.write_parquet(ROSTERS_PARQUET)
    return out


if ROSTERS_PARQUET.exists():
    roster = pl.read_parquet(ROSTERS_PARQUET)
    print(f"Loaded cached snapshot: {ROSTERS_PARQUET}")
else:
    roster = _pull_rosters()
    print(f"Pulled + cached snapshot: {ROSTERS_PARQUET}")

print(f"  teams: {roster['team_abbr'].n_unique()}   players: {roster.height}")
roster.head(3)

Loaded cached snapshot: data/2025-26/reference/rosters.parquet
  teams: 30   players: 530


player_id,player_name,team_name,team_abbr,jersey_number,position,age,experience,birth_date,school
i64,str,str,str,str,str,i64,str,date,str
1642933,"""Keshon Gilbert""","""Atlanta Hawks""","""ATL""",null,"""G""",22,"""R""",2003-06-30,"""Iowa State"""
1642484,"""RayJ Dennis""","""Atlanta Hawks""","""ATL""","""00""","""G""",25,"""1""",2001-03-30,"""Baylor"""
1630228,"""Jonathan Kuminga""","""Atlanta Hawks""","""ATL""","""0""","""F""",23,"""4""",2002-10-06,"""NBA G League Ignite"""


### 2.2 Reconcile the carry-overs

Join the 2024-25 config (all 111 tracked players) to the snapshot on `player_id`
and bucket each one:

- same — still on the same team; carry it as-is.
- moved — different team; re-assign from the snapshot.
- off-roster — no 2025-26 roster row; keep tracked with `roster_team: null`.
  Off-roster isn't no-discourse (Ben Simmons still drives threads), so nothing is
  dropped — the worth-classifying call comes later.

The drift guard flags any team whose config spelling differs from the snapshot's
(which would fake a move); it should be empty.

In [7]:
# §2.2 — Carry-over reconciliation. The 2024-25 config (authoritative "who we
# track", incl. any zero-attribution player a filtered parquet would drop) LEFT
# JOIN the snapshot ON player_id -> same / moved / off-roster. By id, not name,
# so diacritic / "Jr." drift can't fake a miss.
from utils.player_config import load_player_metadata
from IPython.display import display

_meta = load_player_metadata()  # active season (2024-25) = the carry-over set
carryover = pl.DataFrame(
    {
        "player": list(_meta),
        "player_id": [m["player_id"] for m in _meta.values()],
        "team_2024_25": [m["team"] for m in _meta.values()],
    }
)

# Guard: a config franchise spelling absent from the snapshot would fake a move.
spelling_drift = set(carryover["team_2024_25"]) - set(roster["team_name"])

_j = carryover.join(
    roster.select("player_id", "player_name", "team_name", "experience"),
    on="player_id",
    how="left",
)
off_roster = _j.filter(pl.col("team_name").is_null()).sort("player")
_present = _j.filter(pl.col("team_name").is_not_null())
moved = _present.filter(pl.col("team_2024_25") != pl.col("team_name")).sort("player")
same = _present.filter(pl.col("team_2024_25") == pl.col("team_name"))

print(
    f"Carry-overs: {carryover.height}   same: {same.height}   "
    f"moved: {moved.height}   off-roster: {off_roster.height}"
)
print(f"team-spelling drift (want empty): {spelling_drift or '{}'}")

with pl.Config(tbl_rows=40):
    print("\nMOVED (old -> new):")
    display(moved.select("player", "team_2024_25", "team_name"))
    print("OFF-ROSTER (no 2025-26 roster row -> kept, roster_team = null):")
    display(off_roster.select("player", "team_2024_25"))

Carry-overs: 111   same: 75   moved: 32   off-roster: 4
team-spelling drift (want empty): {}

MOVED (old -> new):


player,team_2024_25,team_name
str,str,str
"""Al Horford""","""Boston Celtics""","""Golden State Warriors"""
"""Anthony Davis""","""Dallas Mavericks""","""Washington Wizards"""
"""Bradley Beal""","""Phoenix Suns""","""Los Angeles Clippers"""
"""Brook Lopez""","""Milwaukee Bucks""","""Los Angeles Clippers"""
"""Buddy Hield""","""Golden State Warriors""","""Atlanta Hawks"""
"""Caris LeVert""","""Atlanta Hawks""","""Detroit Pistons"""
"""Clint Capela""","""Atlanta Hawks""","""Houston Rockets"""
"""D'Angelo Russell""","""Brooklyn Nets""","""Washington Wizards"""
"""Damian Lillard""","""Milwaukee Bucks""","""Portland Trail Blazers"""


OFF-ROSTER (no 2025-26 roster row -> kept, roster_team = null):


player,team_2024_25
str,str
"""Ben Simmons""","""Los Angeles Clippers"""
"""Chris Paul""","""San Antonio Spurs"""
"""Lonzo Ball""","""Chicago Bulls"""
"""Patrick Beverley""","""Milwaukee Bucks"""


In [8]:
# §2.2 decision — v2 inclusion is discourse-driven, NOT roster-gated: we drop NO
# carry-over at config-build time. Off-roster players (no 2025-26 roster row) are
# kept tracked with roster_team = None — off-roster != no discourse (Ben Simmons'
# pro-fishing saga, Chris Paul's HOF farewell still drive r/NBA threads). Whether
# a thin-discourse player is "worth classifying" is decided DOWNSTREAM — after the
# player-mention filter, before batch submission — on real 2025-26 comment volume.
TEAMLESS = {
    # off-roster carry-overs kept with roster_team = None (why kept / why no team)
    "Ben Simmons": "off-roster but heavy r/NBA discourse (pro-fishing saga, 'weirdest star')",
    "Patrick Beverley": "off-roster (overseas) but still surfaces in discourse",
    "Chris Paul": "HOF farewell; Clippers -> waived -> Raptors -> waived -> retired",
    "Lonzo Ball": "traded to Utah Feb 2026, waived same day, then a free agent",
}
# player_id + headshot carry over from the 2024-25 config; only team/conference
# go null. Assembly (§2.4) sets roster_team from the snapshot for everyone else
# and None for TEAMLESS.
assert set(TEAMLESS) == set(off_roster["player"]), (
    "every off-roster carry-over must be accounted for (all kept teamless)"
)
print(f"dropped at config time: 0   kept teamless (off-roster): {len(TEAMLESS)}")
print(f"carry-overs retained: {carryover.height} / {carryover.height}")

dropped at config time: 0   kept teamless (off-roster): 4
carry-overs retained: 111 / 111


## 2.3 Additions

### Part A — mined candidates

The 26 mined players from §1.4 come as names; we need their `player_id` and `team`.
This reverses §2.2: match each by normalized name (lowercase, strip diacritics /
punctuation / suffixes) against the snapshot to recover the id.

- resolved — one snapshot match; take its id + team.
- unrostered — no roster row; add anyway with `roster_team: null`.
- ambiguous — multiple matches; flag for a manual id.

Aliases come in Step 3; this pass is only player + team/id.

In [9]:
# §2.3 Part A — resolve the 26 mined candidates (names) to snapshot ids/teams.
# Reverse of §2.2: no player_id yet, so match by NORMALIZED name to *get* it.
import unicodedata

_SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}


def _norm_name(name: str) -> str:
    """Lowercase; drop diacritics; JOIN initials/contractions (T.J. -> tj, so it
    matches a periods-less 'TJ' — nba_api is inconsistent); strip Jr./Sr./II."""
    s = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    s = s.replace(".", "").replace("'", "")  # join: "T.J." -> "TJ", "De'Aaron" -> "DeAaron"
    s = s.replace("-", " ").replace(",", " ")
    return " ".join(t for t in s.lower().split() if t not in _SUFFIXES)


# Normalized-name -> snapshot rows (a list so accidental collisions surface).
_lookup: dict[str, list[dict]] = {}
for _row in roster.iter_rows(named=True):
    _lookup.setdefault(_norm_name(_row["player_name"]), []).append(_row)

mined_resolved, mined_flagged = [], []
for name, mentions, tier, note in NEW_PLAYERS:
    matches = _lookup.get(_norm_name(name), [])
    if len(matches) == 1:
        m = matches[0]
        mined_resolved.append(
            {
                "player": name,
                "player_id": m["player_id"],
                "team": m["team_name"],
                "team_abbr": m["team_abbr"],
                "exp": m["experience"],
                "tier": tier,
            }
        )
    else:
        mined_flagged.append(
            {
                "player": name,
                "tier": tier,
                "status": "unrostered" if not matches else f"ambiguous({len(matches)})",
                "snapshot_names": [x["player_name"] for x in matches],
            }
        )

print(
    f"Mined candidates: {len(NEW_PLAYERS)}   "
    f"resolved: {len(mined_resolved)}   flagged: {len(mined_flagged)}"
)
with pl.Config(tbl_rows=-1, fmt_str_lengths=60):
    display(pl.DataFrame(mined_resolved).sort("team_abbr"))
    if mined_flagged:
        print("FLAGGED — unrostered => add teamless; ambiguous => pin id by hand:")
        display(pl.DataFrame(mined_flagged))

Mined candidates: 26   resolved: 26   flagged: 0


player,player_id,team,team_abbr,exp,tier
str,i64,str,str,str,str
"""Luka Garza""",1630568,"""Boston Celtics""","""BOS""","""4""","""verify"""
"""Zach Collins""",1628380,"""Chicago Bulls""","""CHI""","""8""","""verify"""
"""Cooper Flagg""",1642843,"""Dallas Mavericks""","""DAL""","""R""","""verify"""
"""Dwight Powell""",203939,"""Dallas Mavericks""","""DAL""","""11""","""verify"""
"""Dereck Lively""",1641726,"""Dallas Mavericks""","""DAL""","""2""","""clear"""
"""Max Christie""",1631108,"""Dallas Mavericks""","""DAL""","""3""","""verify"""
"""Christian Braun""",1631128,"""Denver Nuggets""","""DEN""","""3""","""clear"""
"""Brandin Podziemski""",1641764,"""Golden State Warriors""","""GSW""","""2""","""clear"""
"""Steven Adams""",203500,"""Houston Rockets""","""HOU""","""12""","""clear"""


In [10]:
# §2.3 Part A decision — which mined candidates to ADD (the curated keep list).
# Mined adds are borderline-discourse (v1 200-540 mentions), so the keep/drop call
# is made by hand. The excluded set is kept below so nothing drops silently, and the
# config-time choice is still backstopped by the downstream prune.
MINED_KEEP = [
    "Cooper Flagg", "Christian Braun", "Brandin Podziemski", "Steven Adams",
    "Aaron Nesmith", "Andrew Nembhard", "TJ McConnell", "Bennedict Mathurin",
    "Jaxson Hayes", "Kentavious Caldwell-Pope", "Davion Mitchell",
    "Kevin Porter Jr", "Kyle Kuzma", "Jaden McDaniels", "Isaiah Hartenstein",
    "Ajay Mitchell", "Stephon Castle", "RJ Barrett",
]
# Excluded mined candidates — deep role players / low expected 2025-26 discourse.
# (Jaylen Wells later re-added via the Part B per-team pass -> MEM.)
MINED_EXCLUDED = [
    "Luka Garza", "Zach Collins", "Dwight Powell", "Dereck Lively",
    "Max Christie", "Obi Toppin", "Jaylen Wells", "AJ Green",
]
# Volume-risk note: the Pacers adds (Nembhard, Nesmith, McConnell) ride a 50->19
# win collapse; expect depressed 2025-26 discourse -> revisit at the downstream
# worth-classifying prune, not here.

_resolved_names = {r["player"] for r in mined_resolved}
assert set(MINED_KEEP) | set(MINED_EXCLUDED) == _resolved_names, (
    "keep + excluded must partition the resolved mined set exactly"
)
assert not (set(MINED_KEEP) & set(MINED_EXCLUDED)), "keep / excluded must be disjoint"

mined_adds = [r for r in mined_resolved if r["player"] in set(MINED_KEEP)]
print(
    f"mined keep: {len(MINED_KEEP)}   excluded: {len(MINED_EXCLUDED)}   "
    f"(of {len(mined_resolved)} resolved)"
)
with pl.Config(tbl_rows=-1):
    display(pl.DataFrame(mined_adds).sort("team_abbr"))

mined keep: 18   excluded: 8   (of 26 resolved)


player,player_id,team,team_abbr,exp,tier
str,i64,str,str,str,str
"""Cooper Flagg""",1642843,"""Dallas Mavericks""","""DAL""","""R""","""verify"""
"""Christian Braun""",1631128,"""Denver Nuggets""","""DEN""","""3""","""clear"""
"""Brandin Podziemski""",1641764,"""Golden State Warriors""","""GSW""","""2""","""clear"""
"""Steven Adams""",203500,"""Houston Rockets""","""HOU""","""12""","""clear"""
"""Andrew Nembhard""",1629614,"""Indiana Pacers""","""IND""","""3""","""verify"""
"""Aaron Nesmith""",1630174,"""Indiana Pacers""","""IND""","""5""","""clear"""
"""TJ McConnell""",204456,"""Indiana Pacers""","""IND""","""10""","""clear"""
"""Bennedict Mathurin""",1631097,"""Los Angeles Clippers""","""LAC""","""3""","""clear"""
"""Jaxson Hayes""",1629637,"""Los Angeles Lakers""","""LAL""","""6""","""clear"""


### Part B — rookies and young players

Part A only catches players with 2024-25 discourse, so it misses the new rookie
class and quiet sophomores. This pass is roster-driven: anti-join the snapshot
against everyone already tracked to get the untracked rostered players, surfaced
by `age` / `experience`.

- `untracked` — all untracked rostered players.
- `show_untracked("OKC")` — per-team walkthrough at any experience level.
- `young_board` — league-wide rookies + sophomores (`exp` R / 1) as a draft list.

Picks go in `PART_B_KEEP_BY_TEAM`, organized by team.

In [11]:
# §2.3 Part B — untracked rostered players (the rookie / young-player net).
# Tracked so far = carry-overs (all 111) + Part-A mined keeps. Anti-join the
# snapshot to get everyone on a 2025-26 roster we don't yet track.
tracked_ids = set(carryover["player_id"]) | {r["player_id"] for r in mined_adds}

untracked = (
    roster.filter(~pl.col("player_id").is_in(tracked_ids))
    .select("player_id", "player_name", "team_abbr", "position", "age", "experience")
    .sort(["experience", "age"])
)


def show_untracked(team_abbr: str) -> None:
    """Per-team untracked-roster walkthrough (your OKC query, tracked-set aware).
    Shows every experience level, not just young players."""
    with pl.Config(tbl_rows=-1):
        display(untracked.filter(pl.col("team_abbr") == team_abbr.upper()))


# Young-player board: R = rookie, "1" = sophomore. R-only would miss named 2024
# draftees (McCain / Sarr / Sheppard are sophomores), so include exp "1".
young_board = untracked.filter(pl.col("experience").is_in(["R", "1"])).sort(
    ["experience", "team_abbr", "age"]
)

print(f"tracked so far: {len(tracked_ids)}   untracked rostered: {untracked.height}")
print(f'Young-player board (exp R=rookie / 1=sophomore): {young_board.height} players')
with pl.Config(tbl_rows=-1):
    display(young_board)

tracked so far: 129   untracked rostered: 405
Young-player board (exp R=rookie / 1=sophomore): 167 players


player_id,player_name,team_abbr,position,age,experience
i64,str,str,str,i64,str
1642258,"""Zaccharie Risacher""","""ATL""","""F""",21,"""1"""
1642484,"""RayJ Dennis""","""ATL""","""G""",25,"""1"""
1630811,"""Keaton Wallace""","""ATL""","""G""",27,"""1"""
1630623,"""Tyson Etienne""","""BKN""","""G""",26,"""1"""
1631248,"""Baylor Scheierman""","""BOS""","""G""",25,"""1"""
1642275,"""Tidjane Salaün""","""CHA""","""F""",20,"""1"""
1641790,"""PJ Hall""","""CHA""","""C""",24,"""1"""
1641810,"""Antonio Reeves""","""CHA""","""G""",25,"""1"""
1641824,"""Matas Buzelis""","""CHI""","""F""",21,"""1"""


In [12]:
# §2.3 Part B picks — your per-team curation (rookies/young + role players you
# judged discourse-worthy), organized by team. Resolved to snapshot id/team; the
# team always comes from the snapshot (the bucket key is just your filing). A
# bucket != snapshot team is surfaced below. (Julian Champagnie filed to SAS — a
# Spur; neither Champagnie is on POR, Justin is on WAS.)
PART_B_KEEP_BY_TEAM = {
    "ATL": ["Zaccharie Risacher", "CJ McCollum", "Dyson Daniels", "Jalen Johnson", "Onyeka Okongwu", "Nickeil Alexander-Walker", "Gabe Vincent"],
    "BKN": [],
    "BOS": ["Hugo González", "Payton Pritchard"],
    "CHA": ["Kon Knueppel", "Coby White", "Brandon Miller", "Grant Williams", "Miles Bridges", "Moussa Diabaté"],
    "CHI": ["Rob Dillingham", "Matas Buzelis", "Isaac Okoro", "Collin Sexton", "Anfernee Simons"],
    "CLE": ["Dennis Schröder", "Dean Wade", "Max Strus", "Jarrett Allen"],
    "DAL": ["P.J. Washington", "Daniel Gafford"],
    "DEN": ["Tim Hardaway Jr.", "Peyton Watson", "Cameron Johnson", "Bruce Brown"],
    "DET": ["Jalen Duren", "Isaiah Stewart", "Duncan Robinson", "Kevin Huerter", "Paul Reed", "Ronald Holland II"],
    "GSW": ["Moses Moody", "Gary Payton II"],
    "HOU": ["Reed Sheppard", "Jabari Smith Jr.", "Tari Eason"],
    "IND": [],
    "LAC": ["Nicolas Batum", "John Collins", "Bogdan Bogdanović"],
    "LAL": ["Dalton Knecht"],
    "MEM": ["GG Jackson", "Jaylen Wells"],
    "MIA": ["Norman Powell", "Jaime Jaquez Jr."],
    "MIL": ["Gary Trent Jr.", "Ryan Rollins", "Thanasis Antetokounmpo"],
    "MIN": ["Bones Hyland"],
    "NOP": ["Jeremiah Fears", "Derik Queen"],
    "NYK": ["Jordan Clarkson", "Jose Alvarado", "Landry Shamet"],
    "OKC": ["Jared McCain", "Nikola Topić", "Cason Wallace", "Jaylin Williams"],
    "ORL": ["Anthony Black", "Jalen Suggs", "Wendell Carter Jr."],
    "PHI": ["VJ Edgecombe", "Kelly Oubre Jr.", "Andre Drummond", "Kyle Lowry"],
    "PHX": ["Grayson Allen"],
    "POR": ["Deni Avdija", "Shaedon Sharpe", "Donovan Clingan", "Jerami Grant", "Toumani Camara"],
    "SAC": ["Maxime Raynaud", "De'Andre Hunter"],
    "SAS": ["Dylan Harper", "Carter Bryant", "Devin Vassell", "Keldon Johnson", "Julian Champagnie"],
    "TOR": ["Collin Murray-Boyles", "Gradey Dick", "Immanuel Quickley"],
    "UTA": ["Ace Bailey", "Keyonte George", "Kevin Love", "Walker Kessler"],
    "WAS": ["Alex Sarr"],
}
PART_B_KEEP = [n for names in PART_B_KEEP_BY_TEAM.values() for n in names]

partb_resolved, partb_flagged = [], []
_seen_ids: set[int] = set()
for bucket, names in PART_B_KEEP_BY_TEAM.items():
    for name in names:
        matches = _lookup.get(_norm_name(name), [])
        if len(matches) != 1:
            partb_flagged.append({"player": name, "bucket": bucket,
                "status": "unrostered" if not matches else f"ambiguous({len(matches)})"})
            continue
        m = matches[0]
        if m["player_id"] in tracked_ids:
            partb_flagged.append({"player": name, "bucket": bucket, "status": "already-tracked (dup)"})
            continue
        if m["player_id"] in _seen_ids:
            partb_flagged.append({"player": name, "bucket": bucket, "status": "dup within Part B"})
            continue
        _seen_ids.add(m["player_id"])
        rec = {"player": name, "player_id": m["player_id"], "team": m["team_name"],
               "team_abbr": m["team_abbr"], "exp": m["experience"], "bucket": bucket}
        if m["team_abbr"] != bucket:
            rec["bucket_mismatch"] = True
        partb_resolved.append(rec)

_mismatch = [r for r in partb_resolved if r.get("bucket_mismatch")]
print(f"Part B picks: {len(PART_B_KEEP)}   resolved: {len(partb_resolved)}   "
      f"flagged: {len(partb_flagged)}   bucket!=snapshot: {len(_mismatch)}")
if partb_flagged:
    print("FLAGGED:", partb_flagged)
if _mismatch:
    print("BUCKET MISMATCH:", [(r["player"], r["bucket"], r["team_abbr"]) for r in _mismatch])
with pl.Config(tbl_rows=-1):
    display(pl.DataFrame([{k: v for k, v in r.items() if k != "bucket_mismatch"}
                          for r in partb_resolved]).sort("team_abbr"))

Part B picks: 90   resolved: 90   flagged: 0   bucket!=snapshot: 0


player,player_id,team,team_abbr,exp,bucket
str,i64,str,str,str,str
"""Zaccharie Risacher""",1642258,"""Atlanta Hawks""","""ATL""","""1""","""ATL"""
"""CJ McCollum""",203468,"""Atlanta Hawks""","""ATL""","""12""","""ATL"""
"""Dyson Daniels""",1630700,"""Atlanta Hawks""","""ATL""","""3""","""ATL"""
"""Jalen Johnson""",1630552,"""Atlanta Hawks""","""ATL""","""4""","""ATL"""
"""Onyeka Okongwu""",1630168,"""Atlanta Hawks""","""ATL""","""5""","""ATL"""
"""Nickeil Alexander-Walker""",1629638,"""Atlanta Hawks""","""ATL""","""6""","""ATL"""
"""Gabe Vincent""",1629216,"""Atlanta Hawks""","""ATL""","""6""","""ATL"""
"""Hugo González""",1642864,"""Boston Celtics""","""BOS""","""R""","""BOS"""
"""Payton Pritchard""",1630202,"""Boston Celtics""","""BOS""","""5""","""BOS"""


In [13]:
# §2.3 Part B decision — adds locked. Integrity asserts: every pick resolves to
# exactly one rostered player, no flags, no duplicate player_id, no overlap with
# already-tracked (carry-overs + Part-A keeps).
partb_adds = partb_resolved
assert not partb_flagged, f"resolve Part B flags first: {partb_flagged}"
assert len({r["player_id"] for r in partb_adds}) == len(partb_adds), "duplicate player_id in Part B"
assert not ({r["player_id"] for r in partb_adds} & tracked_ids), "Part B overlaps already-tracked"

_total = 111 + len(mined_adds) + len(partb_adds)
print(f"Part B adds: {len(partb_adds)}")
print(f"config total: 111 carry-overs + {len(mined_adds)} mined + {len(partb_adds)} Part B = {_total}")

Part B adds: 90
config total: 111 carry-overs + 18 mined + 90 Part B = 219


## 2.4 Assemble and write the config

Build `players.yaml` from the three groups and write it (blank line between entries
for readability):

- Carry-overs: aliases verbatim from 2024-25 (+ §1.4 gap fixes), team refreshed
  from the snapshot.
- New players: snapshot facts (team, conference, `player_id`, headshot) + full-name
  and diacritic floor aliases.
- Surname pass: add a bare surname only when it's safe to substring-match — unique
  across all 219, unclaimed, and not a dictionary word or too short. The rest are
  flagged for the Step 3 round.

The file is reloaded and asserted (counts, required fields, headshot↔id, teamless
nulls, unique ids). The Step 3 alias decisions and the §1.4 short_aliases sync also
live in this cell — `ALIAS_OVERRIDES`, `EXTRA_ALIASES`, `SHORT_ALIAS_ADD`.

In [14]:
# §2.4 Assembly + §3 surname pass — build config/2025-26/players.yaml.
import collections
import unicodedata
import yaml
from pathlib import Path

from utils.player_config import load_player_config
from utils.team_config import load_team_config

HEADSHOT = "https://cdn.nba.com/headshots/nba/latest/1040x760/{}.png"
_SUF = {"jr", "sr", "ii", "iii", "iv", "v"}

team_cfg = load_team_config()
conf_of = {t: info["conference"] for t, info in team_cfg.items()}
v1_aliases, _ = load_player_config()
team_of = {r["player_id"]: r["team_name"] for r in roster.iter_rows(named=True)}
assert not (set(team_of.values()) - set(conf_of)), "snapshot team missing from teams.yaml"

GAP_ALIAS = {p: v[0] for p, v in ALIAS_GAPS.items()}  # §1.4 gaps -> carry-overs
assert set(GAP_ALIAS) <= set(v1_aliases), "ALIAS_GAPS names must be carry-overs"


def _ascii(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()


def _floor_alias(name):  # full-name floor + ASCII variant for diacritics
    base = " ".join(name.replace(".", "").lower().split())
    asc = _ascii(base)
    return [base] if asc == base else [base, asc]


def _entry(player_id, team, aliases):
    return {"team": team, "conference": conf_of.get(team) if team else None,
            "player_id": int(player_id), "headshot_url": HEADSHOT.format(player_id),
            "aliases": aliases}


def _surname(name):
    toks = [t for t in name.replace(".", "").split() if t.lower() not in _SUF]
    return toks[-1].lower() if toks else name.lower()


players = {}
for row in carryover.iter_rows(named=True):
    name, pid = row["player"], row["player_id"]
    team = None if name in TEAMLESS else team_of.get(pid)
    aliases = list(v1_aliases[name])
    if name in GAP_ALIAS and GAP_ALIAS[name] not in aliases:
        aliases.append(GAP_ALIAS[name])
    players[name] = _entry(pid, team, aliases)

new_order = [rec["player"] for rec in mined_adds + partb_adds]
for rec in mined_adds + partb_adds:
    players[rec["player"]] = _entry(rec["player_id"], rec["team"], _floor_alias(rec["player"]))

n_new = len(new_order)
assert len(players) == 111 + n_new, "name collision merged two players"

# --- §3 collision-aware surname pass (new players only) --------------------
# Add a bare surname ONLY when safe to substring-match: unique across all 219,
# not already owned, and not a common English word / too short. Else FLAG for the
# nba-superfan round (which also sanity-reviews the auto-adds for substring traps).
try:
    with open("/usr/share/dict/words") as _f:
        _WORDS = {w.strip().lower() for w in _f}
except OSError:
    _WORDS = {"love", "black", "white", "green", "brown", "wade", "hart", "smart",
              "moody", "king", "reed", "wells", "dick", "allen"}

_surname_players = collections.defaultdict(list)
for nm in players:
    _surname_players[_surname(nm)].append(nm)
_owners = collections.defaultdict(set)
for nm, d in players.items():
    _owners[nm.lower()].add(nm)
    for a in d["aliases"]:
        _owners[a.lower()].add(nm)

surname_added, surname_flagged = [], []
for nm in new_order:
    s = _surname(nm)
    if len(_surname_players[s]) > 1:
        surname_flagged.append({"player": nm, "surname": s, "reason": "shared surname",
                                "with": [x for x in _surname_players[s] if x != nm]})
    elif _owners.get(s, set()) - {nm}:
        surname_flagged.append({"player": nm, "surname": s, "reason": "alias collision",
                                "with": sorted(_owners[s] - {nm})})
    elif s in _WORDS or len(s) <= 3:
        surname_flagged.append({"player": nm, "surname": s, "reason": "common-word/short -> word-boundary"})
    else:
        if s not in players[nm]["aliases"]:
            players[nm]["aliases"].append(s)
        _owners[s].add(nm)
        surname_added.append({"player": nm, "surname": s})

# Step 3 — user-approved alias additions (deterministic; nba-superfan round adds
# more). jb -> Jaylen Brown (now the clear owner over Brunson); §1.4 surname
# recoveries brooks/ag/ar. All word-boundary-guarded via short_aliases below.
EXTRA_ALIASES = {
    "Jaylen Brown": ["jb"],
    "Dillon Brooks": ["brooks"],
    "Aaron Gordon": ["ag"],
    "Austin Reaves": ["ar"],
}
assert set(EXTRA_ALIASES) <= set(players), "EXTRA_ALIASES targets a non-tracked player"
for _p, _aa in EXTRA_ALIASES.items():
    for _a in _aa:
        if _a not in players[_p]["aliases"]:
            players[_p]["aliases"].append(_a)

# Step 3 nba-superfan round — Category 1 (rookies): finalized aliases. Replaces
# each player's list — e.g. drops the Kobe-magnet bare "bryant" off Carter Bryant,
# adds slang ("the maine event", etc.). Risky tokens -> SHORT_ALIAS_ADD.
ALIAS_OVERRIDES = {
    "Hugo González": ["hugo gonzález", "hugo gonzalez", "gonzález", "hugo", "gonzalez"],
    "Kon Knueppel": ["kon knueppel", "knueppel", "kon", "knuppel"],
    "Cooper Flagg": ["cooper flagg", "flagg", "coop", "cooper", "the maine event"],
    "Jeremiah Fears": ["jeremiah fears", "fears", "jeremiah"],
    "Derik Queen": ["derik queen", "derik", "the big easy", "the louisiana purchase"],
    "Nikola Topić": ["nikola topić", "nikola topic", "topić"],
    "VJ Edgecombe": ["vj edgecombe", "edgecombe", "vj", "v.j.", "edgecomb"],
    "Maxime Raynaud": ["maxime raynaud", "raynaud"],
    "Carter Bryant": ["carter bryant", "cb"],
    "Dylan Harper": ["dylan harper", "harper", "dylan"],
    "Collin Murray-Boyles": ["collin murray-boyles", "murray-boyles"],
    "Ace Bailey": ["ace bailey", "bailey", "ace"],
    # Category 2 (sophomores)
    "Zaccharie Risacher": ["zaccharie risacher", "risacher", "rza", "zacch"],
    "Matas Buzelis": ["matas buzelis", "buzelis", "matas", "buzi vert"],
    "Rob Dillingham": ["rob dillingham", "dillingham"],
    "Ronald Holland II": ["ronald holland", "ron holland", "holland"],
    "Reed Sheppard": ["reed sheppard", "sheppard", "reed", "shep"],
    "Dalton Knecht": ["dalton knecht", "knecht", "dalton k", "dk"],
    "Jaylen Wells": ["jaylen wells", "wells"],
    "Jared McCain": ["jared mccain", "mccain", "j-mac", "jmac", "jared"],
    "Ajay Mitchell": ["ajay mitchell", "ajay", "belgian waffle"],
    "Donovan Clingan": ["donovan clingan", "clingan", "cling kong"],
    "Stephon Castle": ["stephon castle", "stephon", "castle"],
    "Alex Sarr": ["alex sarr", "sarr"],
    # Category 3 (veterans, manual round)
    "CJ McCollum": ["cj", "mccollum"],
    "Nickeil Alexander-Walker": ["nickeil", "alexander walker", "alexander-walker"],
    "Coby White": ["coby"],
    "Moussa Diabaté": ["diabate", "diabaté", "moussa"],
    "Dennis Schröder": ["schröder", "schroder", "schroeder", "dennis the menace"],
    "Paul Reed": ["b-ball paul", "bball paul", "paul reed"],
    "Isaiah Stewart": ["isaiah stewart", "beef stew", "stew", "stewart"],
    "Duncan Robinson": ["duncan robinson", "jimmy neutron", "sheen"],
    "Kevin Huerter": ["red velvet", "huerter"],
    "Gary Payton II": ["gary payton ii", "gp2", "young glove", "the mitton", "payton ii"],
    "Kevin Porter Jr": ["kevin porter jr", "kpj"],
    "Jabari Smith Jr.": ["jabari smith", "jabari"],
    "John Collins": ["john collins", "the baptist", "dunkin deacon"],
    "Jaime Jaquez Jr.": ["jaquez", "jaime", "juan wick"],
    "Ryan Rollins": ["rollins", "rylo", "lightskin blade"],
    "Thanasis Antetokounmpo": ["thanasis", "the greek streak", "thanasty"],
    "Jose Alvarado": ["jose", "alvarado", "alverado", "grand theft alvarado"],
    "Cason Wallace": ["cason wallace", "wallace", "c-wall", "caso", "caydub"],
    "Jaylin Williams": ["jaylin williams", "j-will", "jwill", "jay will"],
    "Kelly Oubre Jr.": ["kelly oubre", "oubre", "wave papi", "tsunami papi"],
    "Deni Avdija": ["deni", "avdija"],
    "Gradey Dick": ["gradey", "grady", "dick"],
    "Immanuel Quickley": ["quickley", "himmanuel", "nesquick"],
    "Keyonte George": ["keyonte", "keynote", "threeyonte", "kg3"],
    "Kevin Love": ["kevin love", "k-love", "k love"],
    "Keldon Johnson": ["keldon"],
}
assert set(ALIAS_OVERRIDES) <= set(players), "ALIAS_OVERRIDES targets a non-tracked player"
for _p, _al in ALIAS_OVERRIDES.items():
    players[_p]["aliases"] = list(_al)

# --- sort by last name, write (blank line between entries) ------------------
def _lastname(name):
    toks = [t for t in name.replace(".", "").split() if t.lower() not in _SUF]
    return (toks[-1].lower() if toks else name.lower(), name.lower())


players = {k: players[k] for k in sorted(players, key=_lastname)}
v1_short = yaml.safe_load(Path("config/2024-25/players.yaml").read_text())["short_aliases"]
SHORT_ALIAS_ADD = [
    "jb", "ag", "ar", "brooks", "sharpe",                            # earlier Step 3
    "kon", "vj", "cb", "ace", "coop", "cooper", "harper", "bailey",  # rookie round
    "gonzalez", "fears", "hugo", "dylan",
    "rza", "matas", "holland", "reed", "shep", "dk", "wells",        # sophomore round
    "mccain", "jmac", "jared", "ajay", "stephon", "castle",
    "cj", "nickeil", "coby", "moussa", "stew", "stewart", "sheen", "gp2",   # vet round
    "kpj", "jabari", "jaime", "rylo", "jose", "wallace", "caso", "jwill",
    "deni", "gradey", "grady", "dick", "keyonte", "keynote", "kg3", "keldon",
]  # word-boundary guards (Step 3)
# §1.4 short_aliases sync: + SHORT_ALIAS_ADD (word-boundary guards), - orphans
# (in v1's short_aliases but bound to no player — inert leftovers).
SHORT_ALIAS_ORPHANS = ["ai", "brown", "green", "james", "jordan", "murray", "paul", "smart", "tt"]
short_aliases = [s for s in dict.fromkeys(v1_short + SHORT_ALIAS_ADD) if s not in SHORT_ALIAS_ORPHANS]
config = {"version": "1.0", "season": "2025-26", "short_aliases": short_aliases, "players": players}

body = yaml.dump(config, sort_keys=False, allow_unicode=True, default_flow_style=False, width=200)
spaced, in_players, first = [], False, True
for line in body.splitlines():
    if line == "players:":
        in_players = True
        spaced.append(line)
        continue
    if in_players and len(line) > 2 and line[:2] == "  " and line[2] != " ":
        if not first:
            spaced.append("")
        first = False
    spaced.append(line)

out = Path("config/2025-26/players.yaml")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(
    "# NBA Player Aliases — 2025-26 (v2 season)\n"
    "# version: MAJOR = roster add/drop, MINOR = alias-only change.\n"
    f"# Carry-overs verbatim (+ §1.4 gaps); {n_new} new players: full-name + diacritic\n"
    "# floor + collision-safe surname. Flagged surnames -> Step 3 nba-superfan round.\n\n"
    + "\n".join(spaced) + "\n"
)
print(f"wrote {out}  ({len(players)} players, {n_new} new)")
print(f"surname pass: +{len(surname_added)} added, {len(surname_flagged)} flagged  (dict {len(_WORDS):,} words)")
with pl.Config(tbl_rows=-1, fmt_str_lengths=90):
    display(pl.DataFrame(surname_flagged))

wrote config/2025-26/players.yaml  (219 players, 108 new)
surname pass: +66 added, 42 flagged  (dict 234,456 words)


player,surname,reason,with
str,str,str,list[str]
"""Stephon Castle""","""castle""","""common-word/short -> word-boundary""",null
"""Kevin Porter Jr""","""porter""","""shared surname""","[""Michael Porter Jr""]"
"""Davion Mitchell""","""mitchell""","""shared surname""","[""Donovan Mitchell"", ""Ajay Mitchell""]"
"""Bennedict Mathurin""","""mathurin""","""common-word/short -> word-boundary""",null
"""RJ Barrett""","""barrett""","""common-word/short -> word-boundary""",null
"""Ajay Mitchell""","""mitchell""","""shared surname""","[""Donovan Mitchell"", ""Davion Mitchell""]"
"""Jalen Johnson""","""johnson""","""shared surname""","[""Cameron Johnson"", ""Keldon Johnson""]"
"""Gabe Vincent""","""vincent""","""common-word/short -> word-boundary""",null
"""Coby White""","""white""","""shared surname""","[""Derrick White""]"


In [15]:
# §2.4 validation — reload the written file and assert integrity.
written = yaml.safe_load(Path("config/2025-26/players.yaml").read_text())
P = written["players"]
assert written["season"] == "2025-26"
assert written["version"] == "1.0"
for _o in SHORT_ALIAS_ORPHANS:
    assert _o not in written["short_aliases"], f"orphan {_o} not removed"
n_expected = 111 + len(mined_adds) + len(partb_adds)
assert len(P) == n_expected, (len(P), n_expected)
for name, d in P.items():
    assert set(d) >= {"team", "conference", "player_id", "headshot_url", "aliases"}, name
    assert d["aliases"], f"empty aliases: {name}"
    assert str(d["player_id"]) in d["headshot_url"], name
    assert (d["conference"] is None) == (d["team"] is None), f"team/conf mismatch: {name}"
for p, (a, *_) in ALIAS_GAPS.items():                  # §1.4 gap aliases landed
    assert a in P[p]["aliases"], f"gap alias '{a}' missing for {p}"
assert "nikola topic" in P["Nikola Topić"]["aliases"], "ASCII floor variant missing"
_teamless = sorted(n for n, d in P.items() if d["team"] is None)
assert set(_teamless) == set(TEAMLESS), _teamless
_ids = [d["player_id"] for d in P.values()]
assert len(_ids) == len(set(_ids)), "duplicate player_id"
for _p, _a in {"Jaylen Brown": "jb", "Dillon Brooks": "brooks", "Aaron Gordon": "ag", "Austin Reaves": "ar"}.items():
    assert _a in P[_p]["aliases"], f"extra alias {_a} missing for {_p}"
for _s in ["jb", "ag", "ar", "brooks", "sharpe"]:
    assert _s in written["short_aliases"], f"short alias {_s} missing"
assert "bryant" not in P["Carter Bryant"]["aliases"], "bare bryant must be dropped (Kobe)"
assert "cb" in P["Carter Bryant"]["aliases"]
for _s in ["kon", "vj", "cb", "ace", "coop", "cooper", "harper", "bailey", "fears"]:
    assert _s in written["short_aliases"], f"rookie short alias {_s} missing"
assert "reed" in P["Reed Sheppard"]["aliases"] and "reed" not in P["Paul Reed"]["aliases"], "reed -> Sheppard"
for _s in ["rza", "matas", "holland", "reed", "shep", "dk", "wells", "mccain", "jmac", "jared", "ajay", "stephon", "castle"]:
    assert _s in written["short_aliases"], f"sophomore short alias {_s} missing"
for _s in ["cj", "gp2", "kpj", "kg3", "dick", "keynote", "jose", "wallace", "jabari", "keldon"]:
    assert _s in written["short_aliases"], f"vet short alias {_s} missing"
print(f"OK: {len(P)} players | {len(_teamless)} teamless | gap aliases in | ascii floor in | ids unique")

OK: 219 players | 4 teamless | gap aliases in | ascii floor in | ids unique


## 3. Alias enrichment

The build gives new players a floor (full name + a safe surname). Nicknames, short
forms, and slang get added here by experience cohort — `nba-superfan` suggests
candidates and the keepers are chosen by hand. Each round lands in `ALIAS_OVERRIDES`
(§2.4 build); risky short/common tokens go to `short_aliases`.

In [16]:
# §3 nba-superfan round — Category 1: rookies (snapshot experience == "R").
# Display rookies + current aliases; nba-superfan proposes r/NBA nicknames for
# them (user has final say). Sophomores / vets are later categories.
_cfg = yaml.safe_load(Path("config/2025-26/players.yaml").read_text())["players"]
rookie_view = pl.DataFrame([
    {"player": rec["player"], "team": rec["team_abbr"],
     "aliases": ", ".join(_cfg[rec["player"]]["aliases"])}
    for rec in sorted(mined_adds + partb_adds, key=lambda r: r["team_abbr"])
    if rec["exp"] == "R"
])
print(f"rookies (exp R) to enrich: {rookie_view.height}")
with pl.Config(tbl_rows=-1, fmt_str_lengths=90):
    display(rookie_view)

rookies (exp R) to enrich: 12


player,team,aliases
str,str,str
"""Hugo González""","""BOS""","""hugo gonzález, hugo gonzalez, gonzález, hugo, gonzalez"""
"""Kon Knueppel""","""CHA""","""kon knueppel, knueppel, kon, knuppel"""
"""Cooper Flagg""","""DAL""","""cooper flagg, flagg, coop, cooper, the maine event"""
"""Jeremiah Fears""","""NOP""","""jeremiah fears, fears, jeremiah"""
"""Derik Queen""","""NOP""","""derik queen, derik, the big easy, the louisiana purchase"""
"""Nikola Topić""","""OKC""","""nikola topić, nikola topic, topić"""
"""VJ Edgecombe""","""PHI""","""vj edgecombe, edgecombe, vj, v.j., edgecomb"""
"""Maxime Raynaud""","""SAC""","""maxime raynaud, raynaud"""
"""Dylan Harper""","""SAS""","""dylan harper, harper, dylan"""


In [17]:
# §3 nba-superfan round — Category 2: sophomores (snapshot experience == "1").
soph_view = pl.DataFrame([
    {"player": rec["player"], "team": rec["team_abbr"],
     "aliases": ", ".join(_cfg[rec["player"]]["aliases"])}
    for rec in sorted(mined_adds + partb_adds, key=lambda r: r["team_abbr"])
    if rec["exp"] == "1"
])
print(f"sophomores (exp 1) to enrich: {soph_view.height}")
with pl.Config(tbl_rows=-1, fmt_str_lengths=90):
    display(soph_view)

sophomores (exp 1) to enrich: 12


player,team,aliases
str,str,str
"""Zaccharie Risacher""","""ATL""","""zaccharie risacher, risacher, rza, zacch"""
"""Rob Dillingham""","""CHI""","""rob dillingham, dillingham"""
"""Matas Buzelis""","""CHI""","""matas buzelis, buzelis, matas, buzi vert"""
"""Ronald Holland II""","""DET""","""ronald holland, ron holland, holland"""
"""Reed Sheppard""","""HOU""","""reed sheppard, sheppard, reed, shep"""
"""Dalton Knecht""","""LAL""","""dalton knecht, knecht, dalton k, dk"""
"""Jaylen Wells""","""MEM""","""jaylen wells, wells"""
"""Ajay Mitchell""","""OKC""","""ajay mitchell, ajay, belgian waffle"""
"""Jared McCain""","""OKC""","""jared mccain, mccain, j-mac, jmac, jared"""


## 4. Per-team coverage check

Tracked players per team in the finished config — a sanity pass that no
market is accidentally empty and the spread is sane.

In [18]:
# §4 — per-team tracked-player count on the written config.
_final = yaml.safe_load(Path("config/2025-26/players.yaml").read_text())["players"]
team_counts = (
    pl.DataFrame([{"team": d["team"] or "(teamless)"} for d in _final.values()])
    .group_by("team").len().rename({"len": "players"}).sort("players", descending=True)
)
print(f"{len(_final)} players across {team_counts.height} groups")
with pl.Config(tbl_rows=-1):
    display(team_counts)

219 players across 31 groups


team,players
str,u32
"""Oklahoma City Thunder""",12
"""Detroit Pistons""",10
"""New York Knicks""",9
"""Houston Rockets""",9
"""Los Angeles Lakers""",9
"""Golden State Warriors""",9
"""San Antonio Spurs""",9
"""Atlanta Hawks""",9
"""Chicago Bulls""",8
